In [1]:
import json
import logging
from dataclasses import replace

import optuna
import pandas as pd
from IPython.display import display

from elo_model.analysis import fingerprint
from elo_model.config import PipelineConfig
from elo_model.evaluation import rating_bands
from elo_model.feature_schema import FEATURE_NAMES
from elo_model.pipeline import (
    analyze_dataset,
    export_sample,
    extract_dataset,
    feature_missingness,
    load_feature_metadata,
    manifest_rows,
    save_feature_dataset,
    split_summary,
)
from elo_model.predictor import RatingPredictor, export_bundle
from elo_model.split import PlayerSplitter
from elo_model.storage import write_json, write_parquet
from elo_model.training import ModelTrainer
from training_data import ChessComConnection, ModelDatasetBuilder
from training_data.preparation import PlayerPoolBuilder

settings = PipelineConfig.expanded()
allow_underfilled_bands = False

logging.basicConfig(
    level=settings.collection.log_level,
    format=settings.collection.log_format,
    force=True,
)
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f"Model output: {settings.artifact_directory.resolve()}")
print(f"Analysis: {settings.engine.time_seconds}s per position, MultiPV={settings.engine.multipv}")
print(f"Features: {len(FEATURE_NAMES)} (schema {settings.features.schema_version})")
print(f"Candidate pool target: {settings.candidate_games:,} unique games before selection")
print(f"Training quota target: {sum(quota.rows for quota in settings.stratification.quotas):,} player rows")
print(f"Collection months: {settings.collection.months}")
print(f"Discovery months: {settings.discovery_months}")
print(f"Discovery enabled: {settings.discovery_enabled}; extra seeds: {settings.discovery_seed_usernames}")
print(f"Stop discovery after {settings.collection.search_stagnation_expansions} expansions without quota improvement")
print(f"Shared raw archive cache: {settings.collection.archive_cache_directory}")
print(f"Username pool: {settings.usernames_path.resolve()}")
print(f"Persistent engine cache: {settings.engine_cache_directory.resolve()}")

C:\Users\mc678p\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model output: C:\Users\mc678p\OneDrive - AT&T Services, Inc\Documents\Tower Strategy & Roaming\Analysis\ELO Estimator\artifacts\rating_model_expanded
Analysis: 0.1s per position, MultiPV=1
Features: 93 (schema 2)
Candidate pool target: 60,000 unique games before selection
Training quota target: 28,000 player rows
Collection months: ((2026, 10), (2026, 9), (2026, 8), (2026, 7), (2026, 6), (2026, 5))
Discovery months: ((2026, 10),)
Discovery enabled: True; extra seeds: ()
Stop discovery after 500 expansions without quota improvement
Shared raw archive cache: data\archive_cache
Username pool: C:\Users\mc678p\OneDrive - AT&T Services, Inc\Documents\Tower Strategy & Roaming\Analysis\ELO Estimator\data\expanded\usernames.json
Persistent engine cache: C:\Users\mc678p\OneDrive - AT&T Services, Inc\Documents\Tower Strategy & Roaming\Analysis\ELO Estimator\data\analysis_cache


In [2]:
settings.data_directory.mkdir(parents=True, exist_ok=True)

collection_settings = {
    "months": [list(month) for month in settings.collection.months],
    "source_player_cap": settings.collection.max_games_per_player,
}

if settings.collection_settings_path.exists():
    previous_settings = json.loads(settings.collection_settings_path.read_text(encoding="utf-8"))
    if any(previous_settings.get(key) != value for key, value in collection_settings.items()):
        raise ValueError(
            "Collection months or source-player cap changed. Restore them or choose "
            "another data_directory; keep engine_cache_directory to reuse evaluations."
        )
elif settings.raw_games_path.exists():
    raise ValueError("Cached games have no collection provenance; choose another data_directory")

usernames = PlayerPoolBuilder(settings).build(progress=True)
collection_settings["usernames_hash"] = fingerprint(usernames)
collection_settings["usernames_count"] = len(usernames)
write_json(settings.collection_settings_path, collection_settings)

connection = ChessComConnection(settings.collection)
builder = ModelDatasetBuilder(connection)

print(f"Ready: {len(usernames):,} saved and discovered usernames")
print("Completed category searches are reused; adding seeds reruns discovery.")

Discovering blitz players: up to 3,000 archive owners


blitz: archive budget:   0%|          | 0/3000 [00:00<?, ?archive/s]

blitz: archive budget:   9%|▉         | 280/3000 [05:08<49:58,  1.10s/archive, discovered=29323, owner=blondiebomber] 


KeyboardInterrupt: 

In [ ]:
logger = logging.getLogger("elo_model.notebook")
cached_games = None

if settings.raw_games_path.exists():
    cached_games = pd.read_parquet(settings.raw_games_path)
    print(f"Reusing {len(cached_games):,} previously collected games")

print(f"Collecting up to {settings.candidate_games:,} candidate games before selection")
games_df = builder.build(
    usernames,
    target_games=settings.candidate_games,
    initial_games=cached_games,
    progress=settings.training.progress,
    checkpoint=lambda frame: write_parquet(settings.raw_games_path, frame),
)

write_parquet(settings.raw_games_path, games_df)

if len(games_df) < settings.candidate_games:
    logger.warning(
        "Candidate pool contains %s/%s games; quotas may be underfilled. "
        "More usernames or archive months may improve coverage.",
        len(games_df), settings.candidate_games,
    )

print(f"Ready: {len(games_df):,} unique clocked candidate games")
print("Rejected games:", dict(builder.rejections))

Reusing 3,000 previously collected games
Ready: 3,000 unique clocked candidate games
Rejected games: {}


In [ ]:
ratings = pd.concat(
    [games_df["white_rating"], games_df["black_rating"]],
    ignore_index=True,
)
categories = pd.concat([games_df["category"], games_df["category"]], ignore_index=True)

print("Collected game categories")
display(games_df["category"].value_counts().rename("games").to_frame())

print("Candidate player-side coverage by rating and time category")
display(pd.crosstab(
    pd.Series(rating_bands(ratings, expanded=True), name="rating_band"),
    categories.rename("category"),
    dropna=False,
))

print("Candidate rows are not training quotas; inspect the selected split next.")
display(games_df.head())

Collected game categories


,games
category,
blitz,2561
rapid,226
bullet,213


Player-side rating coverage


,rows
<1000,2314
1000-1400,522
1400-1800,1264
1800-2200,853
2200+,1047


,game_id,pgn,white_username,black_username,white_rating,black_rating,category,termination,time_control
0,8f9c81dd03f7be38bb5b5d77f906d7e9f009216bcd4527...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat...",Mploki06,gangdang,1452,1423,blitz,resignation,300
1,975cc5c8153c4d7ae22c78c0f751509c77f40833b3f698...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat...",Sqshimi,gangdang,1414,1431,blitz,timeout,300
2,d6d715c88e01301826d76502b4ca5cf6036909f199497e...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat...",gangdang,esketet1,1423,1429,blitz,timeout,300
3,52bc63b6ab03851190381398da4e0e7138a8aa4c3856ea...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat...",gangdang,zxc0956,1414,1418,blitz,checkmate,300
4,1d69f24e0ff2086d328a101b0232fc2cd7cb93aad31ac8...,"[Event ""Live Chess""]\n[Site ""Chess.com""]\n[Dat...",WhyHeyman,gangdang,1433,1406,blitz,resignation,300


In [ ]:
from elo_model.selection import StratifiedGameSelector

coverage_approved = False
splitter = PlayerSplitter(
    fractions=settings.split_fractions,
    seed=settings.training.seed,
    strict_participants=settings.strict_participants,
)

split_candidates = splitter.assign(games_df)
selector = StratifiedGameSelector(settings.stratification, seed=settings.training.seed)
split_games = selector.select(split_candidates)
group_coverage = selector.partition_coverage(split_games)

selection_metadata = {
    "config": settings.stratification.model_dump(mode="json"),
    "seed": settings.training.seed,
    "candidate_games": len(games_df),
    "coverage": selector.coverage.to_dict("records"),
    "partition_coverage": group_coverage.to_dict("records"),
    "allow_underfilled_bands": allow_underfilled_bands,
}
write_json(settings.split_path.with_suffix(".selection.json"), selection_metadata)

if not selector.coverage.empty:
    with pd.option_context("display.max_rows", None):
        display(selector.coverage)
    if selector.coverage["shortfall"].any():
        logger.warning("Some training quotas are underfilled; inspect the selection coverage report.")

coverage = pd.DataFrame.from_dict(split_summary(split_games), orient="index")
display(coverage)
with pd.option_context("display.max_rows", None):
    display(group_coverage)

insufficient_rows = coverage["rows"] < settings.training.min_rows
insufficient_players = coverage["target_players"] < settings.min_players_per_partition
if insufficient_rows.any() or insufficient_players.any():
    raise ValueError("Selected split is too small; collect more candidates before running Stockfish.")

write_parquet(settings.split_path, split_games)
training_groups = group_coverage.loc[group_coverage["partition"] == "train"]
holdout_groups = group_coverage.loc[group_coverage["partition"] != "train"]
shortfalls = not selector.coverage.empty and selector.coverage["shortfall"].any()
sparse_training = (training_groups["players"] < settings.min_training_group_players).any()
sparse_holdouts = (holdout_groups["players"] < settings.min_holdout_group_players).any()

if sparse_holdouts:
    logger.warning(
        "Some validation/test groups have fewer than %s players; their metrics may be unreliable.",
        settings.min_holdout_group_players,
    )

needs_review = settings.min_training_group_players > 0 and (shortfalls or sparse_training)
coverage_approved = not needs_review or allow_underfilled_bands
if not coverage_approved:
    raise ValueError(
        "Coverage needs review before Stockfish. Add discovery seeds or increase the candidate "
        "budget and rerun Cells 2-5. To accept documented shortfalls, set "
        "allow_underfilled_bands = True and rerun this cell."
    )

side_rows = manifest_rows(split_games)
side_rows["rating_band"] = rating_bands(side_rows["rating"], expanded=True)
display(pd.crosstab(side_rows["partition"], side_rows["rating_band"], dropna=False))
display(pd.crosstab(side_rows["partition"], side_rows["category"]))

,rating_band,category,requested_rows,available_rows,available_players,selected_rows,selected_players,shortfall
0,<1000,bullet,100,202,111,100,100,0
1,<1000,blitz,500,1200,648,500,500,0
2,<1000,rapid,100,180,100,100,100,0
3,<1000,classical,100,0,0,0,0,100
4,1000-1400,bullet,100,21,17,21,17,79
5,1000-1400,blitz,500,303,176,279,176,221
6,1000-1400,rapid,100,19,12,19,12,81
7,1000-1400,classical,100,0,0,0,0,100
8,1400-1800,bullet,100,5,2,5,2,95
9,1400-1800,blitz,500,898,456,500,455,0


2026-10-29 09:57:25,222 WARNING [MainThread] Some training quotas are underfilled; inspect the selection coverage report.


,rows,games,target_players
train,2634,2026,1997
validation,364,329,188
test,319,297,181


rating_band,<1000,1000-1400,1400-1800,1800-2200,2200+
partition,,,,,
test,147,34,45,36,57
train,700,319,535,544,536
validation,143,42,76,33,70


category,blitz,bullet,rapid
partition,,,
test,270,29,20
train,2242,161,231
validation,293,39,32


In [ ]:
if not coverage_approved:
    raise ValueError("Review and approve split coverage in Cell 5 before engine analysis.")

engine_metadata = analyze_dataset(
    split_games,
    config=settings.engine,
    cache_directory=settings.engine_cache_directory,
    output=settings.analyzed_path,
    progress=settings.training.progress,
)

analyzed_games = pd.read_parquet(settings.analyzed_path)

print(f"Analysis references saved for {len(analyzed_games):,} games")
print("Rerunning this cell reuses complete cached evaluations")

Analyze games: 100%|██████████| 2652/2652 [04:41<00:00,  9.41game/s, cached=2549, new=103]


Analysis references saved for 2,652 games
Rerunning this cell reuses complete cached evaluations


In [7]:
features_df = extract_dataset(
    analyzed_games,
    cache_directory=settings.engine_cache_directory,
    config=settings.features,
    progress=settings.training.progress,
    engine_metadata=engine_metadata,
)

save_feature_dataset(
    features_df,
    output=settings.features_path,
    engine_metadata=engine_metadata,
    config=settings.features,
)

missingness = pd.Series(feature_missingness(features_df), name="missing_fraction")

display(missingness.sort_values(ascending=False).to_frame())
print(f"Saved {len(features_df):,} player rows with {len(FEATURE_NAMES)} features")

Extract features: 100%|██████████| 2652/2652 [05:05<00:00,  8.68game/s]


,missing_fraction
self_time_trouble_blunder_rate,0.772988
opponent_time_trouble_blunder_rate,0.771480
self_endgame_loss_mean,0.463069
self_endgame_top1_rate,0.463069
self_endgame_blunder_rate,0.463069
...,...
opponent_think_total,0.000000
opponent_time_trouble_share,0.000000
opponent_clock_budget_used,0.000000
opponent_instant_rate,0.000000


Saved 3,317 player rows with 93 features


In [8]:
feature_metadata = load_feature_metadata(settings.features_path)
training_rows = pd.read_parquet(settings.features_path)

trainer = ModelTrainer(settings.training)
result = trainer.fit(training_rows)

result.metadata.update(feature_metadata)
result.metadata["dataset_games"] = len(split_games)
result.metadata["candidate_games"] = len(games_df)
result.metadata["split_fractions"] = list(settings.split_fractions)
result.metadata["strict_participants"] = settings.strict_participants
result.metadata["selection"] = json.loads(
    settings.split_path.with_suffix(".selection.json").read_text(encoding="utf-8")
)

print("Training complete; evaluation used untouched test rows after selection")

Train models: 100%|██████████| 1/1 [00:30<00:00, 30.49s/model]

Training complete; evaluation used untouched test rows after selection


In [ ]:
metric_rows = []

for model_name, report in result.report.items():
    for partition, estimators in report["partitions"].items():
        for estimator_name, metrics in estimators.items():
            metric_rows.append({
                "model": model_name,
                "partition": partition,
                "estimator": estimator_name,
                **metrics["overall"],
            })

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)

for model_name, report in result.report.items():
    for partition in ("validation", "test"):
        metrics = report["partitions"][partition]["lightgbm"]
        print(f"{model_name}: {partition} errors by expanded rating band")
        display(pd.DataFrame.from_dict(metrics["by_expanded_rating_band"], orient="index"))

        category_metrics = [
            {"rating_band": band, "category": category, **values}
            for band, categories in metrics["by_rating_band_category"].items()
            for category, values in categories.items()
        ]
        print(f"{model_name}: {partition} errors by rating band and category")
        display(pd.DataFrame(category_metrics))

    print(f"{model_name}: test errors by time category")
    display(pd.DataFrame.from_dict(
        report["partitions"]["test"]["lightgbm"]["by_category"], orient="index"
    ))

    if "feature_importance" in report:
        importance = pd.DataFrame.from_dict(report["feature_importance"], orient="index")
        display(importance.sort_values("gain", ascending=False))

sample = export_sample(training_rows, result.models)
verification = export_bundle(
    result.models,
    directory=settings.artifact_directory,
    sample=sample,
    metadata=result.metadata,
    report=result.report,
    feature_config=settings.features,
)

print("Export verification:")
display(pd.Series(verification).to_frame("value"))
print(f"Model bundle: {settings.artifact_directory.resolve()}")

,model,partition,estimator,rows,players,mae,rmse,player_macro_mae
0,pooled,train,lightgbm,2634,1997,192.178301,303.489141,202.237789
1,pooled,train,ridge,2634,1997,351.159631,439.190275,365.248188
2,pooled,train,train_median,2634,1997,582.462794,711.759892,620.933399
3,pooled,validation,lightgbm,364,188,337.315847,435.261868,354.379604
4,pooled,validation,ridge,364,188,374.492085,472.068670,397.904941
5,pooled,validation,train_median,364,188,669.777473,797.164796,704.955619
6,pooled,test,lightgbm,319,181,377.597202,486.587094,357.862391
7,pooled,test,ridge,319,181,405.262374,507.026816,383.567940
8,pooled,test,train_median,319,181,713.829154,819.127932,692.199293


pooled: test errors by rating band


,rows,players,mae,rmse,player_macro_mae
<1000,147,82,514.693169,610.199164,507.431273
1000-1400,34,18,283.293795,352.811443,238.236232
1400-1800,45,34,256.160207,339.301832,267.518642
1800-2200,36,20,262.100226,311.751662,223.270617
2200+,57,31,249.101843,371.875348,219.999656


pooled: test errors by time category


,rows,players,mae,rmse,player_macro_mae
blitz,270,162,375.823555,488.253747,364.577521
bullet,29,12,389.677344,487.767993,289.002567
rapid,20,12,384.025222,461.697814,341.133313


,gain,gain_fraction,splits
opponent_accuracy,12507.564099,0.158744,983
self_phase,11441.576416,0.145214,703
self_accuracy,9539.734270,0.121077,818
opponent_phase,8212.159134,0.104227,629
self_clock,7554.450737,0.095880,629
opponent_clock,7465.383416,0.094749,586
opponent_position,5483.915196,0.069601,455
time_control,4733.565624,0.060078,190
self_position,4454.725750,0.056539,382
material,4092.351916,0.051939,337


Export verification:


,value
sample_rows,64.000000
max_absolute_difference,0.000000
warm_single_row_ms,0.152047
native_model_bytes,720059.000000


Model bundle: C:\Users\mc678p\OneDrive - AT&T Services, Inc\Documents\Tower Strategy & Roaming\Analysis\ELO Estimator\artifacts\rating_model


In [10]:
predictor = RatingPredictor(settings.artifact_directory)

held_out = training_rows.loc[training_rows["partition"] == "test"].head(
    settings.preview_rows
)

predictions = predictor.predict(
    held_out.loc[:, list(FEATURE_NAMES)],
    categories=held_out["category"].tolist(),
)

prediction_preview = held_out.loc[:, ["target_username", "category", "rating"]].copy()
prediction_preview["predicted_rating"] = predictions
prediction_preview["absolute_error"] = (
    prediction_preview["predicted_rating"] - prediction_preview["rating"]
).abs()

display(prediction_preview)

print("Inference used the exported native model; no engine analysis was rerun")

,target_username,category,rating,predicted_rating,absolute_error
15,the-angel-tribe,blitz,2201,2018.954020,182.045980
40,riky366,blitz,1664,1787.101074,123.101074
45,paapaa7,blitz,1773,1732.779272,40.220728
58,zojosh9,blitz,1690,845.196063,844.803937
68,incognito6868,blitz,1639,1716.991623,77.991623
88,shbahodir,blitz,2323,2408.421928,85.421928
90,felix-rose,blitz,2274,2488.964846,214.964846
123,broadsky21,blitz,228,759.467202,531.467202
125,kannanchesspoy,blitz,293,695.108260,402.108260
135,rolletta,blitz,1818,1640.484037,177.515963


Inference used the exported native model; no engine analysis was rerun


In [11]:
from datetime import UTC, datetime, timedelta

from elo_model.analysis import AnalysisCache, StockfishAnalyzer
from elo_model.game import GameInput

username = "MCodr"
game_index = 1

current_month = datetime.now(UTC).date().replace(day=1)
previous_month = current_month - timedelta(days=1)

recent_settings = replace(
    settings.collection,
    months=(
        (current_month.year, 10),
        (previous_month.year, 9),
    ),
    max_games_per_player=10,
)

recent_games = ModelDatasetBuilder(
    ChessComConnection(recent_settings)
).build([username], target_games=10)

if not 0 <= game_index < len(recent_games):
    raise ValueError(f"Choose an index from {len(recent_games)} eligible games")

selected = recent_games.iloc[game_index+2]
game = GameInput.from_pgn(selected["pgn"])

with StockfishAnalyzer(
    settings.engine,
    AnalysisCache(settings.engine_cache_directory),
) as analyzer:
    _, analysis = analyzer.analyze(game, progress=True)
    estimates = predictor.predict_game(
        game, analysis, engine_metadata=analyzer.metadata
    )

comparison = pd.DataFrame([
    {
        "player": selected[f"{color}_username"],
        "color": color,
        "actual_rating": selected[f"{color}_rating"],
        "estimated_rating": round(estimates[color]),
    }
    for color in ("white", "black")
])
comparison["absolute_error"] = (
    comparison["estimated_rating"] - comparison["actual_rating"]
).abs()

print(game.headers.get("Date"), game.headers.get("Result"))
display(comparison)

2026.10.29 1-0


,player,color,actual_rating,estimated_rating,absolute_error
0,MCodr,white,1353,1318,35
1,harjae,black,1352,1101,251
